In [1]:
# Client Setup
import boto3

client = boto3.client("bedrock-runtime", region_name="eu-central-1")
model_id = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [5]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    additional_model_fields = {}
    if text_editor:
        additional_model_fields["tools"] = [
            {
                "type": text_editor,
                "name": "str_replace_editor",
            }
        ]

    if thinking:
        additional_model_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    params["additionalModelRequestFields"] = additional_model_fields
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [7]:
messages = []

add_user_message(
    messages,
    "Write a one paragraph guide to recursion",
)

result = chat(messages, thinking=True)

result["parts"]

[{'reasoningContent': {'reasoningText': {'text': 'This is a clever prompt about recursion - a programming concept where a function calls itself. The humor often associated with recursion is that to understand recursion, you must first understand recursion (which is itself recursive). I should write a helpful, clear paragraph that explains recursion well, and maybe include a subtle nod to the recursive nature of the topic itself.',
    'signature': 'EtEECnAIDxABGAIqQJZY+UVnzgozho4roXDRThnl9lCbgMKBJ6VEwWR4B8bCo3PC9cCVUgpt1QIXG3nEKes8Fj+MQX9Q3N7NzNElbS4yGmNsYXVkZS1zb25uZXQtNC01LTIwMjUwOTI5OABCCHRoaW5raW5nEgx+f8BMzCr5p3+k/1gaDFoDzPhnqZMqMqo/fyIwOW/uwtRHxeCQL6JX5Hr3if/S8SD8GYkitWUzvxd2sBMbfzbmqNQKLSexEXAdEjghKo4Dn4lOTxupSUiwJgu/9sFblHbPciU+KUeZYD46J9nuH9UfEFQaf9GKyT++UVvqV4gu3ktSLfquujuGqC/s3RykvBoQvFgGTe4j+p3ubKDaGpRfnjv0J4HaGBc3v0lYmYwgos4wHtjvBpVk3bbFla3zXZ9IXlUwlPP50fcR7Y44WDF8J23FeCS+TNBl6nT/McXWiO9H7BhUq/trKxMIM2LvgjWaz2ddU9lolX57TDhMvszuug1YZ1ECUwjl251jdX2ZX0UqmLCmbew3cGiLyjiNZjjMWzY